# SWOT data discovery and download from NASA Earthdata

This notebook demonstrates a complete introductory workflow for discovering and downloading **SWOT** products from NASA Earthdata.

The workflow is divided into five main steps:

1. Authenticate with NASA Earthdata using `earthaccess`.
2. Explore SWOT collections available in NASA's Common Metadata Repository (CMR).
3. Draw a rectangular area of interest interactively on a map.
4. Search for SWOT RiverSP granules intersecting that area.
5. Download the selected granules safely, with retries, duplicate protection, temporary folders, and a log file.

> **Training note:** Run the cells in order. The search cell uses the `polygon` variable created when a rectangle is drawn on the interactive map.


## 1. Authenticate with NASA Earthdata

`earthaccess` simplifies authentication and access to NASA Earthdata services.

With `persist=True`, the authentication information is stored locally so that the user usually does not need to authenticate again every time the notebook is restarted.


In [ ]:
import earthaccess

# Open an interactive NASA Earthdata login.
# The credentials/token are persisted locally so they can be reused
# by subsequent cells and future notebook sessions.
auth = earthaccess.login(
    strategy="interactive",
    persist=True
)

# Verify that authentication was successful.
print("Authenticated?", auth.authenticated)


## 2. Explore SWOT collections available in NASA CMR

NASA's **Common Metadata Repository (CMR)** contains metadata describing Earth science datasets.

This cell performs a broad search using the keyword `SWOT`. It is useful in a training course because students can see the available collection names, versions, providers, and titles before choosing a specific `short_name`.


In [ ]:
import requests

# Public NASA Common Metadata Repository (CMR) endpoint used to search collections.
url = "https://cmr.earthdata.nasa.gov/search/collections.json"

# Search for collections containing the keyword "SWOT".
# page_size controls the maximum number of collections returned by this request.
params = {
    "keyword": "SWOT",
    "page_size": 100
}

# Send the query to NASA CMR and convert the JSON response to a Python dictionary.
response = requests.get(url, params=params)
data = response.json()

# Display basic metadata for each SWOT-related collection.
# The short_name is especially important because it is used by earthaccess.search_data().
print("SWOT PRODUCTS FOUND:\n")

for entry in data["feed"]["entry"]:
    short_name = entry.get("short_name", "N/A")
    version = entry.get("version_id", "N/A")
    provider = entry.get("data_center", "N/A")
    title = entry.get("title", "N/A")

    print(f"- {short_name} (v{version}) - {provider}")
    print(f"  ↳ {title}\n")


## 3. Draw the area of interest on an interactive map

Use the **rectangle drawing tool** to define the geographic area to be searched.

When a rectangle is created, its coordinates are converted to a Shapely `Polygon` and stored in the variable `polygon`.

The next cell uses `polygon.bounds`, which returns:

`(min_longitude, min_latitude, max_longitude, max_latitude)`

This is the format expected by the `bounding_box` argument in `earthaccess.search_data()`.


In [1]:
from ipyleaflet import Map, DrawControl
import geopandas as gpd
from shapely.geometry import Polygon
import earthaccess
from IPython.display import display

# Create a global map.
# Students can zoom and pan to the study area before drawing a rectangle.
# The layout values control only the visual size of the map in the notebook.
m = Map(
    center=(0, 0),
    zoom=2,
    layout={"height": "400px", "width": "800px"}
)

# This callback function is automatically executed whenever a drawing event occurs.
def handle_draw(target, action, geo_json):
    global polygon

    # We only need to process newly created geometries.
    if action == "created":

        # Extract the GeoJSON geometry generated by ipyleaflet.
        geometry = geo_json["geometry"]

        # Convert the rectangle coordinates to a Shapely Polygon.
        # The polygon is stored globally because the following cell
        # needs access to its geographic bounds.
        polygon = Polygon(geometry["coordinates"][0])

        print("Polygon drawn:", polygon)
        print("Bounding box:", polygon.bounds)


# Add the drawing control to the map.
# Only the rectangle tool is enabled because the Earthaccess search
# will use the rectangle bounds as a geographic bounding box.
draw_control = DrawControl(
    rectangle={
        "shapeOptions": {
            "color": "#0000FF"
        }
    },
    polyline={},
    polygon={},
    circle={},
    marker={},
    circlemarker={}
)

# Connect the drawing event to the callback function.
draw_control.on_draw(handle_draw)

# Add the toolbar to the map and display it.
m.add_control(draw_control)

# IMPORTANT: draw a rectangle before running the next cell.
display(m)


Map(center=[0, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text'…

## 4. Search for SWOT RiverSP granules

This example searches the **SWOT Level-2 High-Rate RiverSP** collection:

`SWOT_L2_HR_RIVERSP_D`

The search is restricted by a temporal interval, granule names containing `Reach`, and the bounding box drawn on the map.

`results` contains the complete Earthaccess result objects needed for downloading.  
`items` contains only the granule names and is used for inspection/display.


In [2]:
import earthaccess

# Reuse the persisted Earthdata authentication.
# Keeping this login call here makes the cell easier to run independently
# during a training session.
earthaccess.login(persist=True)

# Search for SWOT RiverSP granules.
#
# short_name:
#     Identifies the NASA Earthdata collection.
#
# temporal:
#     Defines the beginning and end of the search period.
#
# granule_name:
#     The wildcard expression '*Reach*' restricts the search to RiverSP Reach files.
#
# bounding_box:
#     polygon.bounds comes from the rectangle drawn in the previous cell and
#     follows the order:
#     (minimum longitude, minimum latitude, maximum longitude, maximum latitude).
results = earthaccess.search_data(
    short_name="SWOT_L2_HR_RIVERSP_D",
    temporal=("2025-01-01", "2026-08-17"),
    granule_name="*Reach*",
    bounding_box=polygon.bounds
)

# Extract only the native granule identifiers for display.
# IMPORTANT: these strings are useful for inspection, but the complete
# objects stored in `results` are required by the download routine.
items = [item["meta"]["native-id"] for item in results]

print(f"Total granules found: {len(items)}")
print("Granules found:")

for item in items:
    print(item)

# Leave the list as the last expression so Jupyter can display it as well.
items


Enter your Earthdata Login username:  daniel.moreira
Enter your Earthdata password:  ········


Total granules found: 956
Granules found:
SWOT_L2_HR_RiverSP_Reach_026_243_AS_20250101T122005_20250101T123304_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_256_AS_20250101T231058_20250101T231853_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_271_AS_20250102T121946_20250102T123325_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_284_AS_20250102T231159_20250102T231935_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_299_AS_20250103T121805_20250103T123406_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_312_AS_20250103T231240_20250103T232024_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_327_AS_20250104T121926_20250104T123357_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_340_AS_20250104T231301_20250104T232140_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_355_AS_20250105T122107_20250105T123328_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_368_AS_20250105T231332_20250105T232431_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_383_AS_20250106T122400_20250106T123319_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_396_AS_20250106T231413_20250106T232533_PGD0_01
SWOT_L2_HR_RiverSP_Reach_026_411_AS_20250107T1

['SWOT_L2_HR_RiverSP_Reach_026_243_AS_20250101T122005_20250101T123304_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_256_AS_20250101T231058_20250101T231853_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_271_AS_20250102T121946_20250102T123325_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_284_AS_20250102T231159_20250102T231935_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_299_AS_20250103T121805_20250103T123406_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_312_AS_20250103T231240_20250103T232024_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_327_AS_20250104T121926_20250104T123357_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_340_AS_20250104T231301_20250104T232140_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_355_AS_20250105T122107_20250105T123328_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_368_AS_20250105T231332_20250105T232431_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_383_AS_20250106T122400_20250106T123319_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_396_AS_20250106T231413_20250106T232533_PGD0_01',
 'SWOT_L2_HR_RiverSP_Reach_026_411_AS_20

## 5. Download the granules safely

The final section is intentionally more robust than a minimal `earthaccess.download()` example.

It demonstrates several useful practices:

- create a platform-independent output path with `Path.home()`;
- keep the complete Earthaccess result objects for download;
- optionally limit the number of files for testing;
- remove duplicate granules before downloading;
- download each granule into an isolated temporary folder;
- retry failed downloads;
- avoid overwriting existing files;
- compare file sizes when duplicate filenames occur;
- write a text log for troubleshooting;
- show one progress bar for the complete job.

The download directory is created under the current user's home directory, so the same code works on Windows, Linux, and macOS.


In [4]:
import os
import sys
import time
import shutil
import inspect
import contextlib

from pathlib import Path
from datetime import datetime
from contextlib import contextmanager, ExitStack
from unittest.mock import patch

import earthaccess

from IPython.display import display


# ============================================================
# 1. EARTHDATA LOGIN
# ============================================================

# Reuse the authentication previously stored by earthaccess.
earthaccess.login(persist=True)


# ============================================================
# 2. USER CONFIGURATION
# ============================================================

# Platform-independent output directory.
#
# Examples:
#
# Windows:
# C:\Users\username\swot_course\vietnam_lakes
#
# Linux:
# /home/username/swot_course/vietnam_lakes
#
# macOS:
# /Users/username/swot_course/vietnam_lakes
#
# Path.home() automatically finds the current user's home directory,
# making the notebook portable between operating systems.
DOWNLOAD_DIR = (
    Path.home()
    / "swot_course"
    / "vietnam_river"
)

DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Download directory:", DOWNLOAD_DIR)


# Complete Earthaccess search results.
#
# `results` must come from a previous earthaccess.search_data() call.
FILE_RESULTS = list(results)


# Optional limit for testing.
#
# None -> process all search results
# 10   -> process only the first 10 results
MAX_DOWNLOADS = None
# MAX_DOWNLOADS = 10

if MAX_DOWNLOADS is not None:
    FILE_RESULTS = FILE_RESULTS[:MAX_DOWNLOADS]


# Maximum number of attempts for each granule.
MAX_RETRIES = 3

# Waiting time before retrying a failed download.
SLEEP_SECONDS = 3


# Refresh the Jupyter status only every N seconds.
#
# This is intentionally time-based instead of file-based.
# Even when downloading tens of thousands of granules,
# the browser receives very few display updates.
STATUS_UPDATE_SECONDS = 5.0


# Maximum length of the filename displayed in Jupyter.
MAX_DISPLAY_NAME_LENGTH = 100


# Temporary downloads are stored here before being moved
# to the final directory.
TMP_ROOT = (
    DOWNLOAD_DIR
    / "_tmp_earthaccess_downloads"
)

TMP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# Detailed Earthaccess messages and errors are written here.
LOG_FILE = (
    DOWNLOAD_DIR
    / "download_log_compact_safe.txt"
)


# Keep failed temporary folders?
#
# False is recommended for large production downloads
# because failed temporary directories can accumulate.
KEEP_FAILED_TMP = False


# ============================================================
# 3. BASIC HELPER FUNCTIONS
# ============================================================

def write_log(message):
    """
    Append a timestamped message to the log file.
    """

    now = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            f"[{now}] {message}\n"
        )


def get_native_id(item):
    """
    Return the NASA native-id associated with an
    Earthaccess search result.
    """

    try:

        return str(
            item["meta"]["native-id"]
        )

    except Exception:

        return str(item)


def short_display_name(item):
    """
    Return a compact granule name for the Jupyter status line.
    """

    name = get_native_id(item)

    if len(name) > MAX_DISPLAY_NAME_LENGTH:

        return (
            name[:MAX_DISPLAY_NAME_LENGTH - 3]
            + "..."
        )

    return name


def safe_unique_path(dst_path):
    """
    Generate a unique filename if a file with the same name
    already exists but has a different size.

    Existing files are never overwritten.
    """

    dst_path = Path(dst_path)

    if not dst_path.exists():
        return dst_path


    parent = dst_path.parent

    # Preserve compound extensions when possible.
    suffixes = "".join(
        dst_path.suffixes
    )

    if suffixes:

        base_name = (
            dst_path.name[:-len(suffixes)]
        )

    else:

        base_name = dst_path.name
        suffixes = ""


    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )


    counter = 1

    while True:

        candidate = (
            parent
            / (
                f"{base_name}_duplicate_"
                f"{timestamp}_{counter}"
                f"{suffixes}"
            )
        )

        if not candidate.exists():
            return candidate

        counter += 1


# ============================================================
# 4. SILENT TQDM REPLACEMENT
# ============================================================

class SilentTqdm:
    """
    Minimal replacement for tqdm.

    It is temporarily injected inside Earthaccess modules
    so that internal progress bars do not create widgets
    or visual output in Jupyter.
    """

    def __init__(
        self,
        iterable=None,
        *args,
        **kwargs
    ):

        self.iterable = iterable

        self.total = kwargs.get(
            "total",
            None
        )

        self.n = 0
        self.disable = True


    def __iter__(self):

        if self.iterable is None:
            return iter([])

        return iter(
            self.iterable
        )


    def __enter__(self):
        return self


    def __exit__(
        self,
        *args
    ):
        return False


    def update(
        self,
        n=1,
        *args,
        **kwargs
    ):

        try:
            self.n += n

        except Exception:
            pass


    def close(
        self,
        *args,
        **kwargs
    ):
        pass


    def refresh(
        self,
        *args,
        **kwargs
    ):
        pass


    def display(
        self,
        *args,
        **kwargs
    ):
        pass


    def clear(
        self,
        *args,
        **kwargs
    ):
        pass


    def reset(
        self,
        *args,
        **kwargs
    ):
        self.n = 0


    def set_postfix(
        self,
        *args,
        **kwargs
    ):
        pass


    def set_description(
        self,
        *args,
        **kwargs
    ):
        pass


    def set_description_str(
        self,
        *args,
        **kwargs
    ):
        pass


    def set_postfix_str(
        self,
        *args,
        **kwargs
    ):
        pass


    @staticmethod
    def write(
        *args,
        **kwargs
    ):
        pass


def silent_tqdm(
    iterable=None,
    *args,
    **kwargs
):

    return SilentTqdm(
        iterable,
        *args,
        **kwargs
    )


def quiet_display(
    *args,
    **kwargs
):
    """
    Replacement for IPython display() while Earthaccess
    performs a download.
    """
    return None


def quiet_publish(
    *args,
    **kwargs
):
    """
    Prevent low-level IPython display messages from
    generating notebook output.
    """
    return None


# ============================================================
# 5. SUPPRESS EARTHACCESS NOTEBOOK OUTPUT
# ============================================================

@contextmanager
def suppress_earthaccess_notebook_noise():
    """
    Temporarily suppress progress bars, widgets and display
    messages created internally by Earthaccess.

    The patches exist only while earthaccess.download()
    is running and are restored immediately afterwards.
    """

    patched_module_attributes = []

    with ExitStack() as stack:


        # ----------------------------------------------------
        # Suppress IPython display()
        # ----------------------------------------------------

        try:

            import IPython.display as ipd

            stack.enter_context(
                patch.object(
                    ipd,
                    "display",
                    quiet_display
                )
            )

        except Exception:
            pass


        try:

            import IPython.core.display_functions as display_functions

            stack.enter_context(
                patch.object(
                    display_functions,
                    "display",
                    quiet_display
                )
            )

        except Exception:
            pass


        # ----------------------------------------------------
        # Suppress direct IPython publishing
        # ----------------------------------------------------

        try:

            ip = get_ipython()

            if (
                ip is not None
                and hasattr(
                    ip,
                    "display_pub"
                )
            ):

                stack.enter_context(
                    patch.object(
                        ip.display_pub,
                        "publish",
                        quiet_publish
                    )
                )

        except Exception:
            pass


        # ----------------------------------------------------
        # Replace tqdm already imported inside Earthaccess
        # ----------------------------------------------------

        for module_name, module in list(
            sys.modules.items()
        ):

            if not module_name.startswith(
                "earthaccess"
            ):
                continue


            for (
                attribute_name,
                replacement
            ) in [

                (
                    "tqdm",
                    silent_tqdm
                ),

                (
                    "display",
                    quiet_display
                ),

            ]:

                if hasattr(
                    module,
                    attribute_name
                ):

                    try:

                        old_value = getattr(
                            module,
                            attribute_name
                        )

                        setattr(
                            module,
                            attribute_name,
                            replacement
                        )

                        patched_module_attributes.append(
                            (
                                module,
                                attribute_name,
                                old_value
                            )
                        )

                    except Exception:
                        pass


        try:

            yield


        finally:

            # Restore the original Earthaccess attributes.
            for (
                module,
                attribute_name,
                old_value
            ) in patched_module_attributes:

                try:

                    setattr(
                        module,
                        attribute_name,
                        old_value
                    )

                except Exception:
                    pass


# ============================================================
# 6. EARTHACCESS DOWNLOAD CALL
# ============================================================

def call_earthaccess_download(
    file_result,
    tmp_dir
):
    """
    Call earthaccess.download() while disabling its own
    progress system whenever supported by the installed
    Earthaccess version.
    """

    tmp_dir = Path(
        tmp_dir
    )


    # --------------------------------------------------------
    # Detect supported progress arguments
    # --------------------------------------------------------

    kwargs = {}

    try:

        signature = inspect.signature(
            earthaccess.download
        )


        if (
            "show_progress"
            in signature.parameters
        ):

            kwargs[
                "show_progress"
            ] = False


        if (
            "progress"
            in signature.parameters
        ):

            kwargs[
                "progress"
            ] = False


    except Exception:
        pass


    # --------------------------------------------------------
    # First attempt: Earthaccess result inside a list
    # --------------------------------------------------------

    try:

        earthaccess.download(
            [file_result],
            str(tmp_dir),
            **kwargs
        )

        return


    except TypeError:

        # Older versions may not understand the optional
        # progress arguments.
        try:

            earthaccess.download(
                [file_result],
                str(tmp_dir)
            )

            return

        except Exception:
            pass


    except Exception:
        pass


    # --------------------------------------------------------
    # Second attempt: single result object
    # --------------------------------------------------------

    try:

        earthaccess.download(
            file_result,
            str(tmp_dir),
            **kwargs
        )

        return


    except TypeError:

        earthaccess.download(
            file_result,
            str(tmp_dir)
        )


# ============================================================
# 7. SILENT DOWNLOAD
# ============================================================

def download_silent(
    file_result,
    tmp_dir
):
    """
    Download one granule without producing notebook output.

    stdout and stderr are redirected to the log file, while
    Earthaccess progress bars/widgets are temporarily disabled.
    """

    tmp_dir = Path(
        tmp_dir
    )

    tmp_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    with open(
        LOG_FILE,
        "a",
        encoding="utf-8"
    ) as log_file:


        with (
            contextlib.redirect_stdout(
                log_file
            ),
            contextlib.redirect_stderr(
                log_file
            )
        ):


            with suppress_earthaccess_notebook_noise():

                call_earthaccess_download(
                    file_result,
                    tmp_dir
                )


# ============================================================
# 8. VALIDATE AND MOVE DOWNLOADED FILES
# ============================================================

def list_downloaded_files(
    tmp_dir
):
    """
    Return all files found inside a temporary directory.

    No file extension is assumed.
    """

    tmp_dir = Path(
        tmp_dir
    )

    return [
        path
        for path in tmp_dir.rglob("*")
        if path.is_file()
    ]


def move_valid_downloaded_files(
    tmp_dir,
    final_dir
):
    """
    Validate downloaded files and move them to the final
    directory.

    Rules:
    - zero-byte files are considered invalid;
    - existing files are never overwritten;
    - same name + same size is considered a duplicate;
    - same name + different size preserves both versions.
    """

    tmp_dir = Path(
        tmp_dir
    )

    final_dir = Path(
        final_dir
    )


    downloaded_files = list_downloaded_files(
        tmp_dir
    )


    if not downloaded_files:

        raise RuntimeError(
            "No downloaded files were found "
            "in the temporary directory."
        )


    moved = 0
    skipped = 0


    for src in downloaded_files:


        if not src.exists():
            continue


        src_size = (
            src.stat().st_size
        )


        if src_size <= 0:

            raise RuntimeError(
                f"Downloaded file has zero size: {src}"
            )


        dst = (
            final_dir
            / src.name
        )


        # ----------------------------------------------------
        # Destination file already exists
        # ----------------------------------------------------

        if dst.exists():


            dst_size = (
                dst.stat().st_size
            )


            # Same name and same size:
            # assume the existing file is already complete.
            if dst_size == src_size:

                skipped += 1

                write_log(
                    f"SKIP existing same-size file: {dst}"
                )

                continue


            # Same filename but different size:
            # preserve both files.
            dst = safe_unique_path(
                dst
            )

            write_log(
                "Existing file has a different size. "
                f"Saving as: {dst}"
            )


        # ----------------------------------------------------
        # Move validated file
        # ----------------------------------------------------

        shutil.move(
            str(src),
            str(dst)
        )

        moved += 1

        write_log(
            f"MOVED: {dst}"
        )


    return moved, skipped


# ============================================================
# 9. TEMPORARY DIRECTORY CLEANUP
# ============================================================

def clean_tmp_dir(
    tmp_dir
):
    """
    Delete only temporary directories created under TMP_ROOT.
    """

    tmp_dir = Path(
        tmp_dir
    )


    if (
        tmp_dir.exists()
        and TMP_ROOT in tmp_dir.parents
    ):

        shutil.rmtree(
            tmp_dir,
            ignore_errors=True
        )


# ============================================================
# 10. DOWNLOAD ONE GRANULE SAFELY
# ============================================================

def download_one_safe(
    file_result,
    index
):
    """
    Download one Earthaccess granule with retries.

    Returns
    -------
    "ok"
        At least one new file was downloaded.

    "skip"
        The downloaded file already existed.

    "fail"
        All attempts failed.
    """


    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):


        tmp_dir = (
            TMP_ROOT
            / (
                f"item_{index:06d}_"
                f"attempt_{attempt}_"
                f"{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
            )
        )


        try:

            write_log(
                "------------------------------------------------------------"
            )

            write_log(
                f"Item {index} | "
                f"attempt {attempt}/{MAX_RETRIES}"
            )

            write_log(
                f"Native ID: {get_native_id(file_result)}"
            )

            write_log(
                f"Temporary directory: {tmp_dir}"
            )


            # ------------------------------------------------
            # Download
            # ------------------------------------------------

            download_silent(
                file_result,
                tmp_dir
            )


            # ------------------------------------------------
            # Validate and move
            # ------------------------------------------------

            moved, skipped = move_valid_downloaded_files(
                tmp_dir,
                DOWNLOAD_DIR
            )


            # ------------------------------------------------
            # Remove temporary directory
            # ------------------------------------------------

            clean_tmp_dir(
                tmp_dir
            )


            if moved > 0:
                return "ok"


            if skipped > 0:
                return "skip"


            return "fail"


        except Exception as error:


            write_log(
                f"FAILED item {index}, "
                f"attempt {attempt}: "
                f"{repr(error)}"
            )


            if KEEP_FAILED_TMP:

                write_log(
                    "Temporary directory kept for inspection: "
                    f"{tmp_dir}"
                )


            else:

                clean_tmp_dir(
                    tmp_dir
                )


            if attempt < MAX_RETRIES:

                time.sleep(
                    SLEEP_SECONDS
                )


    return "fail"


# ============================================================
# 11. MAIN DOWNLOAD LOOP
# ============================================================

n_total = len(
    FILE_RESULTS
)

n_ok = 0
n_skip = 0
n_fail = 0


start_time = time.time()

last_status_update = 0.0


# ------------------------------------------------------------
# Write initial information to the log
# ------------------------------------------------------------

write_log(
    "============================================================"
)

write_log(
    "Starting compact safe download"
)

write_log(
    f"Download directory: {DOWNLOAD_DIR}"
)

write_log(
    f"Total items: {n_total}"
)

write_log(
    "No tqdm or persistent Earthaccess progress widgets."
)

write_log(
    "Existing final files will NOT be overwritten."
)

write_log(
    "============================================================"
)


# ------------------------------------------------------------
# Create ONE persistent Jupyter output
# ------------------------------------------------------------

status_display = display(
    f"Starting download | 0/{n_total}",
    display_id=True
)


# ------------------------------------------------------------
# Process search results
# ------------------------------------------------------------

for idx, file_result in enumerate(
    FILE_RESULTS,
    start=1
):


    status = download_one_safe(
        file_result,
        idx
    )


    # --------------------------------------------------------
    # Update counters
    # --------------------------------------------------------

    if status == "ok":

        n_ok += 1


    elif status == "skip":

        n_skip += 1


    else:

        n_fail += 1


    # --------------------------------------------------------
    # Lightweight screen refresh
    # --------------------------------------------------------

    now = time.time()


    should_update = (

        # Refresh at most every few seconds.
        (
            now - last_status_update
            >= STATUS_UPDATE_SECONDS
        )

        # Always show failures immediately.
        or status == "fail"

        # Always show the final result.
        or idx == n_total
    )


    if should_update:


        current_file = short_display_name(
            file_result
        )


        status_text = (
            f"{idx:,}/{n_total:,} | "
            f"OK {n_ok:,} | "
            f"SKIP {n_skip:,} | "
            f"FAIL {n_fail:,} | "
            f"{current_file}"
        )


        # Replace the SAME output.
        # No new notebook line is created.
        status_display.update(
            status_text
        )


        last_status_update = now


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

elapsed_seconds = (
    time.time()
    - start_time
)


# Final status replaces the same Jupyter output.
status_display.update(
    (
        f"DONE | "
        f"{n_total:,}/{n_total:,} | "
        f"OK {n_ok:,} | "
        f"SKIP {n_skip:,} | "
        f"FAIL {n_fail:,} | "
        f"{elapsed_seconds / 60:.1f} min"
    )
)


# ------------------------------------------------------------
# Write final statistics to the log
# ------------------------------------------------------------

write_log(
    "============================================================"
)

write_log(
    "Download finished"
)

write_log(
    f"OK: {n_ok}"
)

write_log(
    f"SKIP: {n_skip}"
)

write_log(
    f"FAIL: {n_fail}"
)

write_log(
    f"Elapsed seconds: {elapsed_seconds:.1f}"
)

write_log(
    "============================================================"
)


# Only a few final lines are printed.
print()

print(
    f"Download directory: {DOWNLOAD_DIR}"
)

print(
    f"Log file: {LOG_FILE}"
)

Download directory: /home/moreira/swot_course/vietnam_river


'2/956 | OK 0 | SKIP 2 | FAIL 0 | SWOT_L2_HR_RiverSP_Reach_026_256_AS_20250101T231058_20250101T231853_PGD0_01'

KeyboardInterrupt: 